In [ ]:
import sys
from pathlib import Path

print("Current dir:", Path.cwd())
sys.path.append(str(Path.cwd().parent))

import config as config
print("Loaded from:", config.__file__)

Setting session and reading data from cloud(s3) 

In [ ]:
from config import get_spark_session, s3_path, BUCKET_NAME
from pyspark.sql.functions import*
spark = get_spark_session("bronze-to-silver")

sellers_df = spark.read.csv(
    s3_path("bronze", "sellers", "olist_sellers_dataset.csv"),
    header=True,
    inferSchema=True
)

sellers_df.show(5)

Understand the data before transforming 

In [ ]:
print(f"number records: {sellers_df.count()}")
print(f"Columns: {sellers_df.columns}")
sellers_df.printSchema()
sellers_df.describe().show()
sellers_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in sellers_df.columns
]).show()

sellers_df.groupBy("seller_id").count().filter("count > 1").show()

In [ ]:
sellers_df.filter(
    (col("seller_city") == "04482255") | (col("seller_city") == "04482255.0") 
).show(truncate=False)


In [ ]:
from pyspark.sql.functions import when, col

sellers_df = sellers_df.withColumn(
    "seller_city",
    when(
        col("seller_city") == "04482255",
        "Unknown"
    ).otherwise(col("seller_city"))
)

In [ ]:
sellers_df.show()